# Customer Churn Prediction - Data Cleaning

This notebook performs:
- Data loading
- Data inspection
- Missing value handling
- Duplicate removal
- Format standardization
- Feature engineering
- Cleaned dataset saving


In [1]:
#Import Libraries
import pandas as pd
import numpy as np

In [26]:
#Load Dataset
df = pd.read_csv("../data/raw/customer_churn.csv")
df.head()

,CustomerID,Age,Gender,Income,SpendingScore,PurchaseAmount,ProductCategory,PaymentMethod,City,State,Country,LastPurchaseDate,IsActive,Returns,DiscountUsed,ReviewScore,Browser,Device,SessionTime,Churn
0,210,41,Female,35167,65,3992,Beauty,UPI,Delhi,DL,IND,2023-01-01,Y,2.0,False,3.315200,NaN,Desktop,253,No
1,328,63,M,20636,38,4968,Clothing,Card,Hyderabad,TN,India,NaN,No,0.0,False,2.189086,Edge,Mobile,214,Yes
2,4212,34,NaN,29854,78,2807,Clothing,UPI,Delhi,MH,IN,01/02/2023,N,2.0,NaN,3.760838,Chrome,Mobile,61,Yes
3,3054,54,NaN,108678,82,3305,Beauty,Cash,Chennai,DL,IND,March 5 2023,NaN,2.0,NaN,3.188211,Chrome,Mobile,167,Yes
4,4620,67,Female,114929,49,4407,NaN,Card,Hyderabad,NaN,IND,2023-01-01,N,NaN,False,3.003102,Firefox,NaN,296,Yes


In [4]:
#Dataset Shape
print("Shape:", df.shape)

Shape: (5200, 20)


In [5]:
#Dataset Info
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5200 entries, 0 to 5199
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   CustomerID        5200 non-null   int64  
 1   Age               5200 non-null   int64  
 2   Gender            4162 non-null   object 
 3   Income            5200 non-null   int64  
 4   SpendingScore     5200 non-null   int64  
 5   PurchaseAmount    5200 non-null   int64  
 6   ProductCategory   4083 non-null   object 
 7   PaymentMethod     4082 non-null   object 
 8   City              4142 non-null   object 
 9   State             4222 non-null   object 
 10  Country           3925 non-null   object 
 11  LastPurchaseDate  3845 non-null   object 
 12  IsActive          4120 non-null   object 
 13  Returns           3913 non-null   float64
 14  DiscountUsed      3534 non-null   object 
 15  ReviewScore       5200 non-null   float64
 16  Browser           3851 non-null   object 


In [6]:
#Null Values
df.isnull().sum()

CustomerID             0
Age                    0
Gender              1038
Income                 0
SpendingScore          0
PurchaseAmount         0
ProductCategory     1117
PaymentMethod       1118
City                1058
State                978
Country             1275
LastPurchaseDate    1355
IsActive            1080
Returns             1287
DiscountUsed        1666
ReviewScore            0
Browser             1349
Device              1281
SessionTime            0
Churn                  0
dtype: int64

In [7]:
#Duplicate Rows
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 200


In [8]:
#Remove Duplicates
df = df.drop_duplicates()
print("Shape after duplicates removed:", df.shape)

Shape after duplicates removed: (5000, 20)


In [9]:
#Standardize Gender
df['Gender'] = df['Gender'].replace({
    'M': 'Male',
    'F': 'Female',
    'male': 'Male',
    'female': 'Female'
})

In [10]:
#Clean Country Names
df['Country'] = df['Country'].astype(str).str.strip().str.title()

df['Country'] = df['Country'].replace({
    'Usa': 'USA',
    'U.S.A': 'USA',
    'United States': 'USA',
    'Uk': 'UK',
    'United Kingdom': 'UK'
})

In [11]:
#Convert LastPurchaseDate to Datetime
df['LastPurchaseDate'] = pd.to_datetime(df['LastPurchaseDate'], errors='coerce')

In [12]:
#Fill Missing Date Values
df['LastPurchaseDate'] = df['LastPurchaseDate'].fillna(df['LastPurchaseDate'].mode()[0])

In [13]:
#Identify Numeric and Categorical Columns
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

print("Numeric Columns:", numeric_cols)
print("Categorical Columns:", categorical_cols)

Numeric Columns: ['CustomerID', 'Age', 'Income', 'SpendingScore', 'PurchaseAmount', 'Returns', 'ReviewScore', 'SessionTime']
Categorical Columns: ['Gender', 'ProductCategory', 'PaymentMethod', 'City', 'State', 'Country', 'IsActive', 'DiscountUsed', 'Browser', 'Device', 'Churn']


In [14]:
#Fill Numeric Missing Values
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())


In [15]:
#Fill Categorical Missing Values
for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

C:\Users\A\AppData\Local\Temp\ipykernel_20348\3455698228.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(df[col].mode()[0])


In [16]:
#Check Missing Values Again
df.isnull().sum()

CustomerID          0
Age                 0
Gender              0
Income              0
SpendingScore       0
PurchaseAmount      0
ProductCategory     0
PaymentMethod       0
City                0
State               0
Country             0
LastPurchaseDate    0
IsActive            0
Returns             0
DiscountUsed        0
ReviewScore         0
Browser             0
Device              0
SessionTime         0
Churn               0
dtype: int64

In [18]:
#Create DaysSinceLastPurchase
reference_date = df['LastPurchaseDate'].max()
df['DaysSinceLastPurchase'] = (reference_date - df['LastPurchaseDate']).dt.days

In [19]:
#Convert Churn to Numeric
df['Churn'] = df['Churn'].replace({
    'Yes': 1,
    'No': 0
})

C:\Users\A\AppData\Local\Temp\ipykernel_20348\3514602565.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['Churn'] = df['Churn'].replace({


In [20]:
#Drop Unnecessary Columns
df = df.drop(columns=['CustomerID', 'LastPurchaseDate'])

In [22]:
#Final Preview
df.head()

,Age,Gender,Income,SpendingScore,PurchaseAmount,ProductCategory,PaymentMethod,City,State,Country,IsActive,Returns,DiscountUsed,ReviewScore,Browser,Device,SessionTime,Churn,DaysSinceLastPurchase
0,41,Female,35167,65,3992,Beauty,UPI,Delhi,DL,Ind,Y,2.0,False,3.315200,Firefox,Desktop,253,0,0
1,63,Male,20636,38,4968,Clothing,Card,Hyderabad,TN,India,No,0.0,False,2.189086,Edge,Mobile,214,1,0
2,34,Female,29854,78,2807,Clothing,UPI,Delhi,MH,In,N,2.0,True,3.760838,Chrome,Mobile,61,1,0
3,54,Female,108678,82,3305,Beauty,Cash,Chennai,DL,Ind,Y,2.0,True,3.188211,Chrome,Mobile,167,1,0
4,67,Female,114929,49,4407,Beauty,Card,Hyderabad,TN,Ind,N,1.0,False,3.003102,Firefox,Tablet,296,1,0


In [23]:
#Save Cleaned Dataset
df.to_csv("../data/processed/cleaned_customer_churn.csv", index=False)
print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!
